In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f

In [2]:
spark = (SparkSession
                .builder
                .master("local[*]")
                .appName("PyS3")
                .config("spark.log.level", "WARN")
                .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
                .config("spark.hadoop.fs.s3a.access.key", "admin")
                .config("spark.hadoop.fs.s3a.secret.key", "password")
                .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
                .config("spark.hadoop.fs.s3a.path.style.access", "true")
                .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
                .config("spark.hadoop.fs.s3a.fast.upload", "true")
                .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4")
                .getOrCreate())

:: loading settings :: url = jar:file:/opt/spark-3.5.8-bin-hadoop3/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/vadim/.ivy2/cache
The jars for the packages stored in: /home/vadim/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-5d937804-37ac-48c9-8135-d8b42f5145d3;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 200ms :: artifacts dl 7ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	-----------------------------------

In [3]:
print(spark.version)

3.5.8


## Создание DataFrame

In [4]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, DoubleType

schema = StructType( [
      StructField("vendor_id", IntegerType(), nullable = True),
      StructField("trip_id", IntegerType(), nullable = True),
      StructField("trip_distance", FloatType(), nullable = True),
      StructField("fare_amount", DoubleType(), nullable = True),
      StructField("store_and_fwd_flag", StringType(), nullable = True)
])

In [5]:
import random

data = list([random.randint(1, 5),\
             random.randint(1, 10000),\
             random.uniform(5.0, 50.0),\
             random.uniform(1.0, 100.0),\
             random.choice(["Y", "N"])\
            ] for x in range(1000000))

In [6]:
df1 = spark.createDataFrame(data, schema)

## Запись

In [7]:
df1.write.mode("overwrite").parquet("s3a://data/df1")

26/07/06 12:55:44 WARN TaskSetManager: Stage 0 contains a task of very large size (3426 KiB). The maximum recommended task size is 1000 KiB.
26/07/06 12:55:45 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers
                                                                                

In [8]:
df1.write.mode("append").parquet("s3a://data/df1")

26/07/06 12:55:50 WARN TaskSetManager: Stage 1 contains a task of very large size (3426 KiB). The maximum recommended task size is 1000 KiB.
26/07/06 12:55:50 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers
                                                                                

## Чтение

In [9]:
df2 = spark.read.parquet("s3a://data/df1")

In [10]:
df2.count()

2000000

In [11]:
df2.show()

+---------+-------+-------------+------------------+------------------+
|vendor_id|trip_id|trip_distance|       fare_amount|store_and_fwd_flag|
+---------+-------+-------------+------------------+------------------+
|        5|   2707|    37.495975|  40.7682516315665|                 Y|
|        5|   7861|    36.521507|11.756095523393277|                 Y|
|        5|   5646|    10.528264|32.983532587762156|                 N|
|        4|   7742|    21.966278|60.672952740380396|                 Y|
|        4|   2806|     5.958365|59.503281103299095|                 Y|
|        3|   6348|    18.344912| 84.05815664231444|                 N|
|        5|     56|     7.427236| 40.18473031552456|                 N|
|        2|   3379|    36.033405|  69.5480713320799|                 Y|
|        5|   1826|    11.407696| 12.44642902211924|                 N|
|        2|    291|    15.490546|  35.4214970054894|                 Y|
|        3|   6819|    24.150904| 69.57075630069502|            